In [20]:
from model_data import GPTModel
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import tiktoken
import torch.nn.functional as F


In [21]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [22]:
#Initiations

context_length = 256
vocab_size = 100256
embedding_dim = 128
batch_size = 4
num_heads = 4
head_dim = embedding_dim // num_heads
dropout = 0.2
n_layers = 5

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    text = f.read()

encoding = tiktoken.get_encoding("cl100k_base")
token_ids = encoding.encode(text)

#splitting data into train and validation
split_idx = int(0.9 * len(token_ids))
val_tokens = token_ids[split_idx:]

In [24]:
#creating input - target pairs
class LLMDataset(Dataset):
    def __init__(self, token_IDs, context_length):
        self.token_IDs = torch.tensor(token_IDs, dtype=torch.long)
        self.context_length = context_length
    
    def __len__(self):
        return len(self.token_IDs) - self.context_length
        
    def __getitem__(self, idx):
        x = self.token_IDs[idx: idx + self.context_length]
        y = self.token_IDs[idx+1: idx + self.context_length+1]

        return x, y

In [25]:
val_dataset = LLMDataset(
    val_tokens,
    context_length
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [26]:
model = GPTModel(vocab_size, embedding_dim, context_length, dropout, num_heads, n_layers)

model.load_state_dict(
    torch.load("gpt_verdict.pth")
)

model = model.to(device)
model.eval()

GPTModel(
  (input_embedding): InputEmbedding(
    (embedding_layer): Embedding(100256, 128)
    (positional_embedding_layer): Embedding(256, 128)
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (trans_block): Sequential(
    (0): TransformerBlock(
      (att): MultiheadAttention(
        (q_layer): Linear(in_features=128, out_features=128, bias=False)
        (k_layer): Linear(in_features=128, out_features=128, bias=False)
        (v_layer): Linear(in_features=128, out_features=128, bias=False)
        (dropout): Dropout(p=0.2, inplace=False)
        (out_proj): Linear(in_features=128, out_features=128, bias=True)
      )
      (layer_norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn_dropout): Dropout(p=0.2, inplace=False)
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=128, bias=True)

In [ ]:
def evaluate(model, dataloader, criterion, device, vocab_size):
    model.eval()
    total_loss = 0.0

    with torch.no_grad(): #with no gradients
        for x_batch, y_batch in dataloader: 

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)

            loss = criterion(logits.view(-1, vocab_size), y_batch.view(-1))

            total_loss += loss.item()

    return total_loss / len(dataloader)

In [28]:
criterion = nn.CrossEntropyLoss()

val_loss = evaluate(
    model,
    val_dataloader,
    criterion,
    device,
    vocab_size
)

print(f"Validation Loss: {val_loss:.4f}")

Validation Loss: 0.0305


In [29]:
import math

perplexity = math.exp(val_loss)

print(f"Perplexity: {perplexity:.2f}")

Perplexity: 1.03


In [ ]:
#Sample validation flow
prompt = "The desultory life of the Riviera lends"
S_encoding = tiktoken.get_encoding('cl100k_base')
S_tokens = S_encoding.encode(prompt)
print(S_tokens)

S_input_ids = torch.tensor(S_tokens)
print(type(S_input_ids))
print(S_input_ids.shape)

S_input_ids = S_input_ids.unsqueeze(0) #add a batch dimension (T,) --> (1,T)
print(S_input_ids.shape)

S_input_ids = S_input_ids.to(device)

with torch.no_grad():
    logits = model(S_input_ids)

print(logits.shape)

S_next_token_logit = logits[:, -1, :]
S_next_token = torch.argmax(S_next_token_logit, dim=-1)
print(S_next_token)

decode = S_encoding.decode([S_next_token.item()])
print(decode)

max_new_tokens = 50
for _ in range(max_new_tokens):
    logits = model(S_input_ids)
    next_token_logits = logits[:, -1, :]
    next_tokens = torch.argmax(next_token_logits, dim=1, keepdim=True)
    input_ids = torch.cat([S_input_ids, next_tokens], dim=1)

[791, 951, 495, 683, 2324, 315, 279, 51768, 26919, 79018]
<class 'torch.Tensor'>
torch.Size([10])
torch.Size([1, 10])
torch.Size([1, 10, 100256])
tensor([5196], device='mps:0')
 itself


In [ ]:
def generate_text(model, prompt, max_new_tokens, temperature=1.0):
    encoding = tiktoken.get_encoding('cl100k_base')
    tokens = encoding.encode(prompt)
    
    input_ids = torch.tensor(tokens)
    input_ids = input_ids.unsqueeze(0) #T --> B, T add a dimension at idx 0
    input_ids = input_ids.to(device)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(input_ids)
            next_token_logits = logits[:, -1, :] #extracting logit of last token in the seq [B, T(-1), V]
            scaled_logits = next_token_logits / temperature # adjusting temp to control the randomness
            probs = F.softmax(scaled_logits,dim=-1) #logits into probs across vocab
            next_tokens = torch.multinomial(probs,num_samples=1) #exactly one token ID
            input_ids = torch.cat([input_ids, next_tokens], dim=1) #Append next token at the end

    generated_text = encoding.decode(input_ids[0].tolist()) 
    return generated_text

In [36]:
prompt1 = "Once upon a time"
prompt2 = "The old man walked into the room and"
prompt3 =  "Artificial intelligence is"

In [38]:
text = generate_text(model, prompt = prompt3, max_new_tokens=50, temperature=1.2)

print(text)

Artificial intelligence is toictions; and having, on my way to Monte Carlo, caught a glimpse of Jack's balustraded terraces between the pines, I had myself borne thither the next day.

I found the couple at tea beneath their palm-trees


In [33]:
# print(len(train_tokens))
print(len(val_tokens))
print(len(val_dataloader))

495
60
